## ANOVA 테스트

In [3]:
import pandas as pd
import numpy as np
from collections import Counter
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 0. 데이터 로드
# ============================================================
df = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\8,9번 파일(최종)\M19_도매_소매업(최종).parquet')
print(f"전체 데이터: {df.shape[0]:,}행 × {df.shape[1]}컬럼")
print(f"회계년도 범위: {df['회계년도'].min()} ~ {df['회계년도'].max()}")
print(f"부실: {df['부실라벨_ICR3년'].sum():,}  |  정상: {(df['부실라벨_ICR3년']==0).sum():,}")


# ============================================================
# 1. 재무비율 변수 정의 (논문 <표 2> 기준, 5개 평가 구분)
# ============================================================
FINANCIAL_VARS = {
    '성장성': {
        '매출액증가율':   '매출액증가율',
        '유형자산증가율': '유형자산증가율',
        '총자산증가율':   '총자산증가율',
    },
    '수익성': {
        '영업이익률': '영업이익률',
        '순이익률':   '순이익률',
        'ROE':        'ROE',
    },
    '레버리지': {
        '부채비율':     '부채비율',
        '차입금의존도': '차입금의존도',
        '자기자본비율': '자기자본비율',
    },
    '이자지급능력 및 현금보유수준': {
        '금융비용부담률': '금융비용부담률',
        '현금비율':       '현금비율',
        '영업CF_총부채':  '영업CF_총부채',   # 이자보상비율 대용
    },
    '생산성 등': {
        '총자산회전율':   '총자산회전율',
        'ROIC':           'ROIC',
        '투하자본회전율': '투하자본회전율',
    },
}

ALL_VARS = {k: v for group in FINANCIAL_VARS.values() for k, v in group.items()}


# ============================================================
# 2. Dickinson(2011) 생애주기 분류
#    논문 동일: 쇄신기 제외, 4단계 (도입기/성장기/성숙기/쇠퇴기)
# ============================================================
def classify_lifecycle(ocf_sign, icf_sign, fcf_sign):
    pattern = f"{ocf_sign}/{icf_sign}/{fcf_sign}"
    lifecycle_map = {
        '+/-/-': '성숙기',
        '+/-/+': '성장기',
        '-/-/+': '도입기',
        '-/+/-': '쇠퇴기',
        '-/+/+': '쇠퇴기',
        '+/+/-': None,   # 쇄신기 → 분석 제외
        '-/-/-': None,
        '+/+/+': None,
    }
    return lifecycle_map.get(pattern, None)


df['OCF_부호'] = df['영업현금흐름비율'].apply(lambda x: '+' if x > 0 else '-')
df['ICF_부호'] = df['투자활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')
df['FCF_부호'] = df['재무활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')
df['생애주기_단년'] = df.apply(
    lambda row: classify_lifecycle(row['OCF_부호'], row['ICF_부호'], row['FCF_부호']),
    axis=1
)


# ============================================================
# 3. 최근 3년 다수결로 생애주기 확정
# ============================================================
def get_majority_lifecycle(group):
    valid = group.sort_values('회계년도')
    valid = valid[valid['생애주기_단년'].notna()]
    if valid.empty:
        return None
    recent = valid.tail(3)
    last = valid.iloc[-1]['생애주기_단년']
    if len(recent) < 2:
        return last
    counter = Counter(recent['생애주기_단년'].tolist())
    mc = counter.most_common()
    return mc[0][0] if mc[0][1] >= 2 else last


lifecycle_result = (
    df.groupby('사업자등록번호')
      .apply(get_majority_lifecycle)
      .reset_index()
)
lifecycle_result.columns = ['사업자등록번호', '생애주기_최종']

last_rows = (
    df.sort_values('회계년도')
      .groupby('사업자등록번호')
      .last()
      .reset_index()
)
last_rows = last_rows.merge(lifecycle_result, on='사업자등록번호', how='left')
last_rows = last_rows[last_rows['생애주기_최종'].notna()].copy()

LIFECYCLE_ORDER = ['도입기', '성장기', '성숙기', '쇠퇴기']
last_rows['생애주기_최종'] = pd.Categorical(
    last_rows['생애주기_최종'], categories=LIFECYCLE_ORDER, ordered=True
)
last_rows = last_rows.sort_values('생애주기_최종')

n_total = len(last_rows)
n_bad   = int(last_rows['부실라벨_ICR3년'].sum())
n_good  = n_total - n_bad

print(f"\n=== 생애주기 분류 완료 (쇄신기 제외) ===")
cnt = last_rows['생애주기_최종'].value_counts()
for lc in LIFECYCLE_ORDER:
    print(f"  {lc}: {cnt.get(lc, 0):,}개  (부실률 {last_rows[last_rows['생애주기_최종']==lc]['부실라벨_ICR3년'].mean()*100:.2f}%)")
print(f"\n전체 기업: {n_total:,}  |  부실: {n_bad:,}  |  정상: {n_good:,}")


# ============================================================
# 4. Winsorize (논문 동일: 하위 1%, 상위 2% 조정)
# ============================================================
def winsorize(series, lower=0.01, upper=0.02):
    lo = series.quantile(lower)
    hi = series.quantile(1 - upper)
    return series.clip(lower=lo, upper=hi)


df_w = last_rows.copy()
for label, col in ALL_VARS.items():
    if col in df_w.columns:
        df_w[col] = winsorize(df_w[col])

print(f"\n=== Winsorize 완료 (하위 1%, 상위 2%) ===")
print(f"  대상 변수: {len(ALL_VARS)}개")


# ============================================================
# 5. Scheffe 사후검정 함수
#    논문: 집단 간 표본수 불균등 → Scheffe 방법 사용
# ============================================================
def scheffe_test(groups_data, group_names, alpha=0.05):
    k         = len(groups_data)
    N         = sum(len(g) for g in groups_data)
    SSW       = sum(((g - g.mean()) ** 2).sum() for g in groups_data)
    df_within = N - k
    MSW       = SSW / df_within

    rows = []
    for i, j in [(i, j) for i in range(k) for j in range(k) if i < j]:
        g1, g2    = groups_data[i], groups_data[j]
        n1, n2    = len(g1), len(g2)
        mean_diff = g1.mean() - g2.mean()

        F_scheffe = (mean_diff ** 2) / (MSW * (1/n1 + 1/n2))
        p_approx  = 1 - stats.f.cdf(F_scheffe / (k - 1), k - 1, df_within)
        sig = '***' if p_approx < 0.001 else '**' if p_approx < 0.01 else '*' if p_approx < 0.05 else 'n.s.'

        rows.append({
            '비교':        f"{group_names[i]} vs {group_names[j]}",
            '평균(A)':     round(g1.mean(), 4),
            '평균(B)':     round(g2.mean(), 4),
            '평균차(A-B)': round(mean_diff, 4),
            'F(Scheffe)':  round(F_scheffe, 4),
            'p값':         round(p_approx, 4),
            '유의성':      sig,
        })
    return pd.DataFrame(rows)


# ============================================================
# 6. ANOVA + Scheffe 사후검정 실행
# ============================================================
print(f"\n{'='*70}")
print("ANOVA + Scheffe 사후검정 결과")
print(f"{'='*70}")

anova_summary_rows = []
scheffe_all        = []

for group_label, var_dict in FINANCIAL_VARS.items():
    print(f"\n\n{'━'*70}")
    print(f"  [{group_label}]")
    print(f"{'━'*70}")

    for var_label, col in var_dict.items():
        if col not in df_w.columns:
            print(f"\n  ⚠ {var_label} ({col}) : 컬럼 없음, 스킵")
            continue

        groups, group_names, desc_rows = [], [], []
        for lc in LIFECYCLE_ORDER:
            sub = df_w[df_w['생애주기_최종'] == lc][col].dropna()
            if len(sub) >= 5:
                groups.append(sub.values)
                group_names.append(lc)
                desc_rows.append({
                    '생애주기': lc, 'N': len(sub),
                    '평균':     round(sub.mean(), 4),
                    '중위수':   round(sub.median(), 4),
                    '표준편차': round(sub.std(), 4),
                })

        if len(groups) < 2:
            print(f"\n  ⚠ {var_label} : 유효 그룹 부족, 스킵")
            continue

        f_stat, p_value = f_oneway(*groups)

        all_vals   = np.concatenate(groups)
        grand_mean = all_vals.mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        ss_total   = ((all_vals - grand_mean) ** 2).sum()
        eta_sq     = ss_between / ss_total if ss_total > 0 else np.nan
        eta_label  = '대' if eta_sq >= 0.14 else '중' if eta_sq >= 0.06 else '소'

        sig_mark = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'n.s.'

        print(f"\n  ▶ {var_label} ({col})")
        print(pd.DataFrame(desc_rows).to_string(index=False))
        print(f"\n    F = {f_stat:.4f}  |  p = {p_value:.6f}  {sig_mark}"
              f"  |  Eta² = {eta_sq:.4f} ({eta_label} 효과)")

        if p_value < 0.05:
            sch = scheffe_test(groups, group_names)
            print(f"\n    [Scheffe 사후검정]")
            print(sch.to_string(index=False))
            sch.insert(0, '변수', var_label)
            sch.insert(0, '평가구분', group_label)
            scheffe_all.append(sch)
        else:
            print(f"    → ANOVA 비유의 → 사후검정 생략")

        anova_summary_rows.append({
            '평가구분': group_label, '변수': var_label,
            'F통계량':  round(f_stat, 4), 'p값': round(p_value, 6),
            '유의성':   sig_mark, 'Eta²': round(eta_sq, 4), '효과크기': eta_label,
        })


# ============================================================
# 7. ANOVA 요약 출력 (논문 <표 5> 형식)
# ============================================================
anova_summary = pd.DataFrame(anova_summary_rows)

print(f"\n\n{'='*70}")
print("ANOVA 요약 (논문 <표 5> 형식)")
print(f"{'='*70}")
print(anova_summary.to_string(index=False))


# ============================================================
# 8. 부실위험 카이제곱 검정 (논문 <표 11> 형식)
# ============================================================
print(f"\n\n{'='*70}")
print("카이제곱 검정 — 생애주기별 부실위험 차이 (논문 <표 11> 형식)")
print(f"{'='*70}")

# 교차표
crosstab = pd.crosstab(
    last_rows['생애주기_최종'],
    last_rows['부실라벨_ICR3년'],
    margins=True
)
crosstab.columns   = ['정상', '부실', '합계']
crosstab['부실률(%)'] = (crosstab['부실'] / crosstab['합계'] * 100).round(2)
print("\n[교차표]")
print(crosstab.to_string())

# 전체 카이제곱
ct_raw            = pd.crosstab(last_rows['생애주기_최종'], last_rows['부실라벨_ICR3년'])
chi2, p_chi2, dof, expected = chi2_contingency(ct_raw)
cramers_v         = np.sqrt(chi2 / (len(last_rows) * (min(ct_raw.shape) - 1)))

print(f"\n[전체 카이제곱 검정]")
print(f"  χ² = {chi2:.4f}  |  자유도 = {dof}  |  p = {p_chi2:.6f}"
      f"  {'***' if p_chi2 < 0.001 else '**' if p_chi2 < 0.01 else '*' if p_chi2 < 0.05 else 'n.s.'}")
print(f"  Cramér's V = {cramers_v:.4f}"
      f"  ({'강한' if cramers_v >= 0.3 else '중간' if cramers_v >= 0.1 else '약한'} 연관성)")

# 생애주기별 개별 카이제곱
print(f"\n[생애주기별 개별 카이제곱]")
print(f"  {'구분':8s}  {'업체수(구성비)':18s}  {'부실률':8s}  {'통계량':12s}  {'df':4s}  {'χ²값':12s}")
print(f"  {'-'*72}")

for lc in LIFECYCLE_ORDER:
    sub  = last_rows[last_rows['생애주기_최종'] == lc]
    rest = last_rows[last_rows['생애주기_최종'] != lc]
    n    = len(sub)
    ratio = n / n_total * 100
    bad_r = sub['부실라벨_ICR3년'].mean() * 100

    ct2 = pd.DataFrame({
        '해당':  [(sub['부실라벨_ICR3년'] == 0).sum(),  sub['부실라벨_ICR3년'].sum()],
        '나머지': [(rest['부실라벨_ICR3년'] == 0).sum(), rest['부실라벨_ICR3년'].sum()],
    })
    try:
        c2, p2, d2, _ = chi2_contingency(ct2)
        sig2 = '***' if p2 < 0.001 else '**' if p2 < 0.01 else '*' if p2 < 0.05 else 'n.s.'
        print(f"  {lc:8s}  {n:6,}({ratio:5.1f}%)      {bad_r:5.2f}%    카이제곱  {d2:4d}  {c2:10.2f} {sig2}")
    except Exception as e:
        print(f"  {lc:8s}  계산 불가: {e}")


# ============================================================
# 9. 결과 저장
# ============================================================
OUTPUT_BASE = r'C:\유비온프로젝트2\corporate-bankruptcy\생애주기 스코어링'

anova_summary.to_csv(
    f'{OUTPUT_BASE}\\lifecycle_anova_summary.csv',
    index=False, encoding='utf-8-sig'
)
crosstab.to_csv(
    f'{OUTPUT_BASE}\\lifecycle_chisquare_crosstab.csv',
    encoding='utf-8-sig'
)
if scheffe_all:
    pd.concat(scheffe_all, ignore_index=True).to_csv(
        f'{OUTPUT_BASE}\\lifecycle_scheffe_detail.csv',
        index=False, encoding='utf-8-sig'
    )

print(f"\n\n✅ 저장 완료")
print(f"   lifecycle_anova_summary.csv    : ANOVA 요약 (F값, p값, Eta²)")
print(f"   lifecycle_scheffe_detail.csv   : Scheffe 사후검정 쌍별 상세")
print(f"   lifecycle_chisquare_crosstab.csv : 카이제곱 교차표")

전체 데이터: 39,908행 × 277컬럼
회계년도 범위: 2012 ~ 2024
부실: 1,509  |  정상: 38,399

=== 생애주기 분류 완료 (쇄신기 제외) ===
  도입기: 1,327개  (부실률 41.15%)
  성장기: 1,005개  (부실률 19.20%)
  성숙기: 2,524개  (부실률 11.33%)
  쇠퇴기: 1,173개  (부실률 36.66%)

전체 기업: 6,029  |  부실: 1,455  |  정상: 4,574

=== Winsorize 완료 (하위 1%, 상위 2%) ===
  대상 변수: 15개

ANOVA + Scheffe 사후검정 결과


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [성장성]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ▶ 매출액증가율 (매출액증가율)
생애주기    N       평균     중위수    표준편차
 도입기 1327  -0.0929 -2.2836 38.5877
 성장기 1005   0.0344 -1.0492 31.9803
 성숙기 2524  -4.0310 -2.8675 29.9521
 쇠퇴기 1173 -11.5673 -8.3998 38.3120

    F = 29.8375  |  p = 0.000000  ***  |  Eta² = 0.0146 (소 효과)

    [Scheffe 사후검정]
        비교   평균(A)    평균(B)  평균차(A-B)  F(Scheffe)     p값  유의성
도입기 vs 성장기 -0.0929   0.0344   -0.1272      0.0080 0.9998 n.s.
도입기 vs 성숙기 -0.0929  -4.0310    3.9381     11.6347 0.0088   **
도입기 vs 쇠퇴기 -0.0929 -11.5673   11.4744     70.7104 0.00

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

# ============================================================
# 0. 데이터 로드
# ============================================================
df = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\8,9번 파일(최종)\M19_도매_소매업(최종).parquet')
print(f"전체 데이터: {df.shape[0]:,}행 × {df.shape[1]}컬럼")


# ============================================================
# 1. Dickinson 생애주기 분류 함수 (동일)
# ============================================================
def classify_lifecycle(ocf_sign, icf_sign, fcf_sign):
    pattern = f"{ocf_sign}/{icf_sign}/{fcf_sign}"
    lifecycle_map = {
        '+/-/-': '성숙기',
        '+/-/+': '성장기',
        '-/-/+': '도입기',
        '-/+/-': '쇠퇴기',
        '-/+/+': '쇠퇴기',
        '+/+/-': '조정기',
        '-/-/-': '조정기',
        '+/+/+': '조정기',
    }
    return lifecycle_map.get(pattern, '조정기')


# ============================================================
# 2. 부호 컬럼 미리 생성 (반복 불필요)
# ============================================================
df['OCF_부호'] = df['영업현금흐름비율'].apply(lambda x: '+' if x > 0 else '-')
df['ICF_부호'] = df['투자활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')
df['FCF_부호'] = df['재무활동으로 인한 현금흐름(요약)(백만원)'].apply(lambda x: '+' if x > 0 else '-')

df['생애주기_단년'] = df.apply(
    lambda row: classify_lifecycle(row['OCF_부호'], row['ICF_부호'], row['FCF_부호']),
    axis=1
)


# ============================================================
# 3. 최근 3년 다수결 생애주기 확정 함수 (동일)
# ============================================================
def get_majority_lifecycle(group):
    group_sorted = group.sort_values('회계년도')
    recent = group_sorted.tail(3)
    last_lifecycle = group_sorted.iloc[-1]['생애주기_단년']
    if len(recent) < 3:
        return last_lifecycle
    counter = Counter(recent['생애주기_단년'].tolist())
    most_common = counter.most_common()
    return most_common[0][0] if most_common[0][1] >= 2 else last_lifecycle


# ============================================================
# 4. WoE 계산 함수 (동일)
# ============================================================
def calculate_woe(df_input, category_col, label_col):
    total_bad  = df_input[label_col].sum()
    total_good = (df_input[label_col] == 0).sum()
    rows = []
    for cat in df_input[category_col].unique():
        sub  = df_input[df_input[category_col] == cat]
        bad  = sub[label_col].sum()
        good = (sub[label_col] == 0).sum()
        bad_rate  = (bad  + 0.5) / (total_bad  + 0.5)
        good_rate = (good + 0.5) / (total_good + 0.5)
        woe = np.log(bad_rate / good_rate)
        iv  = (bad_rate - good_rate) * woe
        rows.append({
            '생애주기':  cat,
            '전체':      len(sub),
            '부실':      int(bad),
            '정상':      int(good),
            '부실률(%)': round(bad / len(sub) * 100, 2),
            '부실비율':  round(bad_rate, 4),
            '정상비율':  round(good_rate, 4),
            'WoE':       round(woe, 4),
            'IV':        round(iv, 4),
        })
    result = pd.DataFrame(rows).sort_values('WoE', ascending=False).reset_index(drop=True)
    result['IV_합계'] = round(result['IV'].sum(), 4)
    return result


# ============================================================
# 5. 연도별 루프 (2015 ~ 2024)
# ============================================================
all_years_woe    = []   # 연도별 WoE 테이블 누적
all_years_scored = []   # 연도별 기업별 점수 누적

BASE_SCORE = 500
PDO        = 20

for year in range(2015, 2025):  # 2015 ~ 2024

    # ── 해당 연도까지 누적 데이터 ──────────────────────────────
    df_cut = df[df['회계년도'] <= year].copy()

    if df_cut.empty:
        print(f"\n[{year}] 데이터 없음, 스킵")
        continue

    # ── 생애주기 확정 ─────────────────────────────────────────
    lifecycle_result = (
        df_cut.groupby('사업자등록번호')
              .apply(get_majority_lifecycle)
              .reset_index()
    )
    lifecycle_result.columns = ['사업자등록번호', '생애주기_최종']

    # ── 마지막 행 추출 + 생애주기 병합 ───────────────────────
    last_rows = (
        df_cut.sort_values('회계년도')
              .groupby('사업자등록번호')
              .last()
              .reset_index()
    )
    last_rows = last_rows.merge(lifecycle_result, on='사업자등록번호', how='left')

    total_bad  = int(last_rows['부실라벨_ICR3년'].sum())
    total_good = int((last_rows['부실라벨_ICR3년'] == 0).sum())

    if total_bad == 0 or total_good == 0:
        print(f"\n[{year}] 부실/정상 한쪽이 0 → WoE 계산 불가, 스킵")
        continue

    # ── WoE 계산 ──────────────────────────────────────────────
    woe_table = calculate_woe(last_rows, '생애주기_최종', '부실라벨_ICR3년')
    woe_table.insert(0, '기준연도', year)

    # ── PDO 변환 ──────────────────────────────────────────────
    BASE_ODDS = total_bad / total_good
    factor    = PDO / np.log(2)
    offset    = BASE_SCORE - factor * np.log(BASE_ODDS)
    woe_table['PDO_점수'] = (offset + factor * woe_table['WoE']).round(2)

    # ── 0~100 스케일링 ────────────────────────────────────────
    pdo_min = woe_table['PDO_점수'].min()
    pdo_max = woe_table['PDO_점수'].max()
    if pdo_max == pdo_min:
        woe_table['위험점수_최종'] = 50.0   # 모두 동점이면 중간값
    else:
        woe_table['위험점수_최종'] = (
            (woe_table['PDO_점수'] - pdo_min) / (pdo_max - pdo_min) * 100
        ).round(1)

    all_years_woe.append(woe_table)

    # ── 기업별 점수 부여 ──────────────────────────────────────
    score_map = dict(zip(woe_table['생애주기'], woe_table['위험점수_최종']))
    last_rows['기준연도']    = year
    last_rows['생애주기_점수'] = last_rows['생애주기_최종'].map(score_map)
    all_years_scored.append(
        last_rows[['기준연도', '사업자등록번호', '회계년도',
                   '생애주기_최종', '생애주기_점수', '부실라벨_ICR3년']]
    )

    print(f"\n[{year}] 기업 {len(last_rows):,}개  |  부실 {total_bad:,}  |  정상 {total_good:,}")
    print(woe_table[['생애주기', '부실률(%)', 'WoE', 'PDO_점수', '위험점수_최종']].to_string(index=False))


# ============================================================
# 6. 결과 합치기 & 저장
# ============================================================
result_woe    = pd.concat(all_years_woe,    ignore_index=True)
result_scored = pd.concat(all_years_scored, ignore_index=True)

result_scored.to_parquet(
    r'C:\유비온프로젝트2\corporate-bankruptcy\생애주기 스코어링\lifecycle_scored_yearly.parquet',
    index=False
)
result_woe.to_csv(
    r'C:\유비온프로젝트2\corporate-bankruptcy\생애주기 스코어링\lifecycle_woe_yearly.csv',
    index=False, encoding='utf-8-sig'
)

print("\n✅ 저장 완료")
print("   - lifecycle_scored_yearly.parquet : 연도별 기업 점수")
print("   - lifecycle_woe_yearly.csv        : 연도별 생애주기 WoE/점수 테이블")

전체 데이터: 39,908행 × 277컬럼

[2015] 기업 3,140개  |  부실 489  |  정상 2,651
생애주기  부실률(%)     WoE  PDO_점수  위험점수_최종
 쇠퇴기   29.46  0.8188  572.40    100.0
 도입기   24.68  0.5763  565.40     84.3
 조정기   15.62  0.0076  548.99     47.6
 성숙기    8.87 -0.6348  530.46      6.1
 성장기    8.10 -0.7297  527.72      0.0

[2016] 기업 3,443개  |  부실 563  |  정상 2,880
생애주기  부실률(%)     WoE  PDO_점수  위험점수_최종
 쇠퇴기   31.42  0.8526  571.70    100.0
 도입기   27.44  0.6613  566.18     87.6
 조정기   15.11 -0.0902  544.49     38.7
 성장기    9.01 -0.6720  527.71      0.9
 성숙기    8.93 -0.6866  527.29      0.0

[2017] 기업 3,766개  |  부실 633  |  정상 3,133
생애주기  부실률(%)     WoE  PDO_점수  위험점수_최종
 쇠퇴기   31.91  0.8422  570.45    100.0
 도입기   28.59  0.6850  565.91     89.7
 조정기   14.66 -0.1585  541.57     34.2
 성장기    9.55 -0.6428  527.60      2.4
 성숙기    9.26 -0.6794  526.54      0.0

[2018] 기업 4,101개  |  부실 723  |  정상 3,378
생애주기  부실률(%)     WoE  PDO_점수  위험점수_최종
 쇠퇴기   31.64  0.7724  566.77    100.0
 도입기   31.08  0.7461  566.01     98.3
 조정기   16.